# Phase 2: Cheetah-run Experiments

**All quantum-inspired methods tested on Cheetah-run environment.**

This notebook loads and analyzes pre-computed results from 5 quantum-inspired world model training approaches:
- **Baseline**: Standard DreamerV3-style training
- **Quantum Tunneling**: QAOA-inspired optimizer for escaping local minima
- **Superposition**: Parallel exploration of multiple training paths
- **Entanglement**: Correlated feature learning with quantum gate-inspired layers
- **Interference Ensemble**: Error correction with ensemble averaging

**Environment**: Cheetah-run (DMControl Suite)
- Observation dimension: 17
- Action dimension: 6
- Task: Fast locomotion

---
## 1. Setup and Imports

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Set up paths
project_root = Path.cwd().parent
results_path = project_root / "experiments" / "results" / "phase2" / "cheetah" / "complete_metrics.json"

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## 2. Load Results Data

In [2]:
# Load the results
with open(results_path, 'r') as f:
    data = json.load(f)

# Extract components
experiment_name = data['experiment']
environment = data['environment']
config = data['config']
summary = data['summary']
raw_results = data['raw_results']

print(f"Experiment: {experiment_name}")
print(f"Environment: {environment['domain']}-{environment['task']} (obs_dim={environment['obs_dim']}, action_dim={environment['action_dim']})")
print(f"\nConfiguration:")
print(f"  - stoch_dim: {config['stoch_dim']}")
print(f"  - deter_dim: {config['deter_dim']}")
print(f"  - hidden_dim: {config['hidden_dim']}")
print(f"  - batch_size: {config['batch_size']}")
print(f"  - seq_len: {config['seq_len']}")
print(f"  - num_steps: {config['num_steps']}")
print(f"  - learning_rate: {config['learning_rate']}")
print(f"  - Seeds: {config['seeds']}")
print(f"\nApproaches tested: {list(summary.keys())}")

Experiment: phase2_cheetah_run
Environment: cheetah-run (obs_dim=17, action_dim=6)

Configuration:
  - stoch_dim: 64
  - deter_dim: 512
  - hidden_dim: 512
  - batch_size: 32
  - seq_len: 20
  - num_steps: 10000
  - learning_rate: 0.0003
  - Seeds: [42, 123, 456, 789, 1024]

Approaches tested: ['baseline', 'quantum_tunneling', 'superposition', 'entanglement', 'interference_ensemble']


---
## 3. Summary Results Table

In [3]:
# Create summary table
print("=============================================================================")
print("                     Cheetah-run World Model Results Summary")
print("=============================================================================")
print()
print(f"{'Approach':<22}{'Test MSE (mean +/- std)':<27}{'Train MSE':<16}{'Time (s)':<12}{'Params'}")
print("-" * 84)

best_approach = None
best_mse = float('inf')

for approach, metrics in summary.items():
    test_mse_mean = metrics['test_obs_mse_mean']
    test_mse_std = metrics['test_obs_mse_std']
    train_mse_mean = metrics['train_obs_mse_mean']
    time_mean = metrics['time_mean']
    num_params = metrics['num_params']
    
    print(f"{approach:<22}{test_mse_mean:.4f} +/- {test_mse_std:.4f}          {train_mse_mean:.4f}          {time_mean:.2f}     {num_params}")
    
    if test_mse_mean < best_mse:
        best_mse = test_mse_mean
        best_approach = approach

baseline_mse = summary['baseline']['test_obs_mse_mean']
improvement = (baseline_mse - best_mse) / baseline_mse * 100

print()
print(f"Best performer: {best_approach} with Test MSE = {best_mse:.4f} +/- {summary[best_approach]['test_obs_mse_std']:.4f}")
print(f"Improvement over baseline: {improvement:.2f}%")

                     Cheetah-run World Model Results Summary

Approach              Test MSE (mean +/- std)    Train MSE       Time (s)    Params
------------------------------------------------------------------------------------
baseline              0.5733 +/- 0.0087          0.5586          820.25      4749587
quantum_tunneling     0.5784 +/- 0.0050          0.5634          824.90      4749587
superposition         2.8575 +/- 0.0618          2.9103          933.80      4749587
entanglement          0.5750 +/- 0.0070          0.5599          994.68      5275539
interference_ensemble 0.3673 +/- 0.0070          0.3566          5173.24     23747941

Best performer: interference_ensemble with Test MSE = 0.3673 +/- 0.0070
Improvement over baseline: 35.94%


---
## 4. Statistical Analysis (Mann-Whitney U Tests)

In [4]:
# Extract raw test MSE values for each approach
approach_values = {}
for result in raw_results:
    approach = result['approach']
    if 'test_obs_mse' in result:  # Skip any errors
        if approach not in approach_values:
            approach_values[approach] = []
        approach_values[approach].append(result['test_obs_mse'])

# Baseline values
baseline_values = approach_values['baseline']

print("=============================================================================")
print("                Statistical Comparison vs Baseline (Mann-Whitney U)")
print("=============================================================================")
print()
print(f"{'Approach':<22}{'Baseline MSE':<16}{'Approach MSE':<16}{'U-stat':<10}{'p-value':<12}{'Significant?'}")
print("-" * 88)

statistical_results = {}
alpha = 0.05 / 4  # Bonferroni correction for 4 comparisons

for approach in ['quantum_tunneling', 'superposition', 'entanglement', 'interference_ensemble']:
    approach_vals = approach_values[approach]
    baseline_mean = np.mean(baseline_values)
    approach_mean = np.mean(approach_vals)
    
    # Mann-Whitney U test
    u_stat, p_value = stats.mannwhitneyu(baseline_values, approach_vals, alternative='two-sided')
    
    significant = "Yes ***" if p_value < alpha else "No"
    
    statistical_results[approach] = {
        'u_stat': u_stat,
        'p_value': p_value,
        'significant': p_value < alpha,
        'better': approach_mean < baseline_mean
    }
    
    print(f"{approach:<22}{baseline_mean:.4f}          {approach_mean:.4f}          {u_stat:<10.1f}{p_value:<12.4f}{significant}")

print()
print(f"Note: *** indicates p < 0.05 (statistically significant difference from baseline)")
print(f"      Using Bonferroni correction: alpha = 0.05/4 = {alpha:.4f}")

print()
print("Key Findings:")
for approach, results in statistical_results.items():
    if results['significant']:
        direction = "BETTER" if results['better'] else "WORSE"
        print(f"  - {approach}: SIGNIFICANTLY {direction} than baseline (p={results['p_value']:.4f})")
    else:
        print(f"  - {approach}: No significant difference (p={results['p_value']:.4f})")

                Statistical Comparison vs Baseline (Mann-Whitney U)

Approach              Baseline MSE    Approach MSE    U-stat    p-value     Significant?
----------------------------------------------------------------------------------------
quantum_tunneling     0.5733          0.5784          9.0       0.5476      No
superposition         0.5733          2.8575          0.0       0.0079      Yes ***
entanglement          0.5733          0.5750          11.0      0.7540      No
interference_ensemble 0.5733          0.3673          0.0       0.0079      Yes ***

Note: *** indicates p < 0.05 (statistically significant difference from baseline)
      Using Bonferroni correction: alpha = 0.05/4 = 0.0125

Key Findings:
  - interference_ensemble: SIGNIFICANTLY BETTER than baseline (p=0.0079)
  - superposition: SIGNIFICANTLY WORSE than baseline (p=0.0079)
  - quantum_tunneling: No significant difference (p=0.5476)
  - entanglement: No significant difference (p=0.7540)


---
## 5. Visualization: Test MSE Comparison

In [5]:
# Prepare data for plotting
approaches = list(summary.keys())
means = [summary[a]['test_obs_mse_mean'] for a in approaches]
stds = [summary[a]['test_obs_mse_std'] for a in approaches]

# Color coding: green for best, red for worst, blue for others
colors = []
for i, (m, a) in enumerate(zip(means, approaches)):
    if a == 'interference_ensemble':  # Best
        colors.append('#2ecc71')  # Green
    elif a == 'superposition':  # Worst
        colors.append('#e74c3c')  # Red
    else:
        colors.append('#3498db')  # Blue

# Create bar chart
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(approaches))
bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='black', linewidth=1.2)

# Add baseline reference line
baseline_mse = summary['baseline']['test_obs_mse_mean']
ax.axhline(y=baseline_mse, color='red', linestyle='--', linewidth=2, label=f'Baseline: {baseline_mse:.3f}')

# Labels and formatting
ax.set_xlabel('Approach', fontsize=14)
ax.set_ylabel('Test Observation MSE', fontsize=14)
ax.set_title('Cheetah-run: World Model Prediction Accuracy\n(Lower is Better)', fontsize=16)
ax.set_xticks(x)
ax.set_xticklabels(['Baseline', 'Quantum\nTunneling', 'Superposition', 'Entanglement', 'Interference\nEnsemble'], 
                   fontsize=11)

# Add value labels on bars
for i, (bar, mean, std) in enumerate(zip(bars, means, stds)):
    height = bar.get_height()
    ax.annotate(f'{mean:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height + std + 0.02),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add significance markers
for i, approach in enumerate(approaches[1:], 1):  # Skip baseline
    if approach in statistical_results and statistical_results[approach]['significant']:
        ax.annotate('***', xy=(i, means[i] + stds[i] + 0.08), ha='center', fontsize=14, color='black')

ax.legend(loc='upper right', fontsize=11)
ax.set_ylim(0, max(means) + max(stds) + 0.3)

plt.tight_layout()
plt.savefig(project_root / "experiments" / "results" / "phase2" / "cheetah" / "test_mse_comparison.png", dpi=150)
plt.show()

print(f"\nFigure saved to: experiments/results/phase2/cheetah/test_mse_comparison.png")


Figure saved to: experiments/results/phase2/cheetah/test_mse_comparison.png


---
## 6. Per-Seed Results Analysis

In [6]:
# Create DataFrame with per-seed results
seed_data = {}
for result in raw_results:
    approach = result['approach']
    if 'test_obs_mse' in result:
        seed = result['seed']
        if seed not in seed_data:
            seed_data[seed] = {}
        seed_data[seed][approach] = result['test_obs_mse']

df_seeds = pd.DataFrame(seed_data).T
df_seeds.index.name = 'Seed'

print("=============================================================================")
print("                         Per-Seed Test MSE Results")
print("=============================================================================")
print()
print(df_seeds.round(4).to_string())

# Calculate coefficient of variation for consistency
print()
print("Consistency Analysis (Coefficient of Variation = std/mean):")
cvs = {}
for approach in df_seeds.columns:
    cv = df_seeds[approach].std() / df_seeds[approach].mean() * 100
    cvs[approach] = cv

most_consistent = min(cvs, key=cvs.get)
for approach, cv in cvs.items():
    marker = "  <-- Most consistent" if approach == most_consistent else ""
    print(f"  {approach:<22} CV = {cv:.2f}%{marker}")

                         Per-Seed Test MSE Results

                   baseline  quantum_tunneling  superposition  entanglement  interference_ensemble
Seed                                                                                               
42                   0.5667             0.5793         2.8138        0.5740                 0.3629
123                  0.5743             0.5803         2.9763        0.5774                 0.3718
456                  0.5778             0.5858         2.8609        0.5822                 0.3614
789                  0.5865             0.5757         2.8185        0.5794                 0.3788
1024                 0.5614             0.5708         2.8181        0.5620                 0.3614

Consistency Analysis (Coefficient of Variation = std/mean):
  baseline:              CV = 1.52%
  quantum_tunneling:     CV = 0.86%  <-- Most consistent
  superposition:         CV = 2.16%
  entanglement:          CV = 1.22%
  interference_ensemble: CV 

---
## 7. Training Time Comparison

In [7]:
print("=============================================================================")
print("                         Training Time Analysis")
print("=============================================================================")
print()
print(f"{'Approach':<22}{'Time (s)':<16}{'Time (min)':<16}{'Relative to Baseline'}")
print("-" * 74)

baseline_time = summary['baseline']['time_mean']

for approach, metrics in summary.items():
    time_s = metrics['time_mean']
    time_min = time_s / 60
    relative = time_s / baseline_time
    
    if approach == 'baseline':
        rel_str = "1.00x"
    elif relative < 1:
        rel_str = f"{relative:.2f}x ({(1-relative)*100:.1f}% faster)"
    else:
        rel_str = f"{relative:.2f}x ({(relative-1)*100:.1f}% slower)"
    
    print(f"{approach:<22}{time_s:<16.2f}{time_min:<16.2f}{rel_str}")

# Efficiency analysis
print()
print("Efficiency Analysis (MSE per second of training):")
efficiencies = {}
for approach, metrics in summary.items():
    efficiency = metrics['test_obs_mse_mean'] / metrics['time_mean']
    efficiencies[approach] = efficiency

best_eff = min(efficiencies, key=efficiencies.get)
worst_eff = max(efficiencies, key=efficiencies.get)

for approach, eff in efficiencies.items():
    marker = ""
    if approach == best_eff:
        marker = "  <-- Best efficiency"
    elif approach == worst_eff:
        marker = "  <-- Worst efficiency"
    print(f"  {approach:<22} {eff:.3e} MSE/s{marker}")

                         Training Time Analysis

Approach              Time (s)        Time (min)      Relative to Baseline
--------------------------------------------------------------------------
baseline              820.25          13.67           1.00x
quantum_tunneling     824.90          13.75           1.01x (0.6% slower)
superposition         933.80          15.56           1.14x (13.8% slower)
entanglement          994.68          16.58           1.21x (21.3% slower)
interference_ensemble 5173.24         86.22           6.31x (530.7% slower)

Efficiency Analysis (MSE per second of training):
  baseline:              6.990e-04 MSE/s
  quantum_tunneling:     7.012e-04 MSE/s
  superposition:         3.060e-03 MSE/s  <-- Worst efficiency
  entanglement:          5.781e-04 MSE/s
  interference_ensemble: 7.101e-05 MSE/s  <-- Best efficiency


---
## 8. Conclusions

In [8]:
print("=============================================================================")
print("                    CHEETAH-RUN EXPERIMENT CONCLUSIONS")
print("=============================================================================")
print()
print("KEY FINDINGS:")
print()

# Best performer
best = 'interference_ensemble'
best_mse = summary[best]['test_obs_mse_mean']
best_std = summary[best]['test_obs_mse_std']
baseline_mse = summary['baseline']['test_obs_mse_mean']
baseline_std = summary['baseline']['test_obs_mse_std']
improvement = (baseline_mse - best_mse) / baseline_mse * 100

print(f"1. BEST PERFORMER: Interference Ensemble")
print(f"   - Test MSE: {best_mse:.4f} +/- {best_std:.4f} (vs baseline {baseline_mse:.4f} +/- {baseline_std:.4f})")
print(f"   - Improvement: {improvement:.2f}% reduction in prediction error")
print(f"   - Statistical significance: p = {statistical_results[best]['p_value']:.4f} (highly significant)")
print(f"   - Trade-off: 6.3x training time, 5x more parameters")
print()

# Worst performer
worst = 'superposition'
worst_mse = summary[worst]['test_obs_mse_mean']
worst_std = summary[worst]['test_obs_mse_std']
degradation = (worst_mse - baseline_mse) / baseline_mse * 100

print(f"2. WORST PERFORMER: Superposition")
print(f"   - Test MSE: {worst_mse:.4f} +/- {worst_std:.4f} ({degradation:.0f}% worse than baseline)")
print(f"   - Statistical significance: p = {statistical_results[worst]['p_value']:.4f} (significantly worse)")
print(f"   - Note: Parallel path exploration severely hurts performance on fast locomotion")
print()

# Non-significant results
print(f"3. NO SIGNIFICANT DIFFERENCE:")
print(f"   - Quantum Tunneling: {summary['quantum_tunneling']['test_obs_mse_mean']:.4f} (p={statistical_results['quantum_tunneling']['p_value']:.4f}) - similar to baseline")
print(f"   - Entanglement: {summary['entanglement']['test_obs_mse_mean']:.4f} (p={statistical_results['entanglement']['p_value']:.4f}) - similar to baseline")
print()

print("COMPARISON WITH WALKER-WALK:")
print()
print("- Interference Ensemble is best in BOTH environments")
print("- Superposition fails in BOTH environments")
print("- Quantum Tunneling and Entanglement show no benefit in either")
print("- Ensemble approach provides larger absolute gains on Walker (43% vs 36%)")
print()

print("PRACTICAL RECOMMENDATIONS:")
print()
print("- For Cheetah-run locomotion tasks:")
print("  * USE Interference Ensemble when accuracy is critical (despite 6x training cost)")
print("  * USE Baseline for fast iteration during development")
print("  * AVOID Superposition (catastrophic performance degradation)")
print()
print("- The consistent success of ensemble averaging across both DMControl environments")
print("  suggests it is a robust choice for locomotion world models.")
print()
print("=============================================================================")

                    CHEETAH-RUN EXPERIMENT CONCLUSIONS

KEY FINDINGS:

1. BEST PERFORMER: Interference Ensemble
   - Test MSE: 0.3673 +/- 0.0070 (vs baseline 0.5733 +/- 0.0087)
   - Improvement: 35.94% reduction in prediction error
   - Statistical significance: p = 0.0079 (highly significant)
   - Trade-off: 6.3x training time, 5x more parameters

2. WORST PERFORMER: Superposition
   - Test MSE: 2.8575 +/- 0.0618 (399% worse than baseline)
   - Statistical significance: p = 0.0079 (significantly worse)
   - Note: Parallel path exploration severely hurts performance on fast locomotion

3. NO SIGNIFICANT DIFFERENCE:
   - Quantum Tunneling: 0.5784 (p=0.5476) - similar to baseline
   - Entanglement: 0.5750 (p=0.7540) - similar to baseline

COMPARISON WITH WALKER-WALK:

- Interference Ensemble is best in BOTH environments
- Superposition fails in BOTH environments
- Quantum Tunneling and Entanglement show no benefit in either
- Ensemble approach provides larger absolute gains on Walker (43